In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM : une Qiskit Function par Qedma
> **Note:** Les Qiskit Functions sont une fonctionnalité expérimentale réservée aux utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.

## Vue d'ensemble
Bien que les unités de traitement quantique aient considérablement progressé ces dernières années, les erreurs dues au bruit et aux imperfections du matériel existant restent un défi central pour les développeurs d'algorithmes quantiques. À mesure que le domaine s'approche de calculs quantiques à l'échelle utilitaire impossibles à vérifier classiquement, des solutions garantissant une suppression du bruit précise deviennent de plus en plus importantes. Pour relever ce défi, Qedma a développé la suppression et l'atténuation d'erreurs quantiques (QESEM), intégrée de façon transparente sur IBM Quantum Platform en tant que [Qiskit Function](/guides/functions).

Avec QESEM, tu peux exécuter tes circuits quantiques sur des QPU bruités pour obtenir des résultats précis et sans erreur, avec des surcoûts en temps QPU très efficaces, proches des limites fondamentales. Pour ce faire, QESEM exploite un ensemble de méthodes propriétaires développées par Qedma pour la caractérisation et la réduction des erreurs. Les techniques de réduction des erreurs comprennent l'optimisation des portes, la transpilation tenant compte du bruit, la suppression d'erreurs (ES) et l'atténuation d'erreurs non biaisée (EM). Grâce à cette combinaison de méthodes basées sur la caractérisation, tu peux obtenir des résultats fiables et sans erreur pour des circuits quantiques génériques de grand volume, ce qui ouvre la voie à des applications autrement inaccessibles.

Pour une description complète des composants sous-jacents ainsi qu'une démonstration à l'échelle utilitaire, consulte l'article [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Description
Tu peux utiliser la fonction QESEM de Qedma pour estimer et exécuter facilement tes circuits avec suppression et atténuation des erreurs, en atteignant des volumes de circuits plus grands et des précisions plus élevées. Pour utiliser QESEM, tu fournis un circuit quantique, un ensemble d'observables à mesurer, une précision statistique cible pour chaque observable, ainsi qu'un QPU choisi. Avant d'exécuter le circuit jusqu'à la précision cible, tu peux estimer le temps QPU nécessaire à partir d'un calcul analytique qui ne nécessite aucune exécution de circuit. Une fois satisfait de l'estimation du temps QPU, tu peux exécuter le circuit avec QESEM.

Lorsque tu exécutes un circuit, QESEM lance un protocole de caractérisation du dispositif adapté à ton circuit, ce qui produit un modèle de bruit fiable pour les erreurs survenant dans le circuit. Sur la base de cette caractérisation, QESEM met d'abord en œuvre une transpilation tenant compte du bruit pour mapper le circuit d'entrée sur un ensemble de qubits et de portes physiques, minimisant ainsi le bruit affectant l'observable cible. Cela inclut les portes nativement disponibles (CX/CZ sur les dispositifs IBM&reg;), ainsi que des portes supplémentaires optimisées par QESEM, formant le jeu de portes étendu de QESEM. QESEM exécute ensuite un ensemble de circuits ES et EM basés sur la caractérisation sur le QPU et collecte leurs résultats de mesure. Ces résultats sont ensuite post-traités classiquement pour fournir une valeur d'espérance non biaisée et une barre d'erreur pour chaque observable, correspondant à la précision demandée.

![Aperçu de Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM a démontré sa capacité à fournir des résultats de haute précision pour une variété d'applications quantiques et sur les plus grands volumes de circuits réalisables aujourd'hui. QESEM offre les fonctionnalités suivantes pour l'utilisateur, illustrées dans la section des benchmarks ci-dessous :
-	**Précision garantie :** QESEM produit des estimations non biaisées des valeurs d'espérance des observables. Sa méthode EM est dotée de garanties théoriques qui — combinées à la caractérisation de pointe de Qedma — assurent que l'atténuation converge vers la sortie du circuit sans bruit, à la précision spécifiée par l'utilisateur. Contrairement à de nombreuses méthodes EM heuristiques sujettes à des erreurs systématiques ou des biais, la précision garantie de QESEM est essentielle pour assurer des résultats fiables pour des circuits et des observables génériques.
-	**Passage à l'échelle sur de grands QPU :** Le temps QPU de QESEM dépend des volumes de circuits, mais est par ailleurs indépendant du nombre de qubits. Qedma a démontré QESEM sur les plus grands dispositifs quantiques disponibles aujourd'hui, notamment les dispositifs IBM Quantum Eagle à 127 qubits et Heron à 133 qubits.
-	**Indépendant de l'application :** QESEM a été démontré sur une variété d'applications, notamment la simulation hamiltonienne, VQE, QAOA et l'estimation d'amplitude. Tu peux saisir n'importe quel circuit quantique et observable à mesurer, et obtenir des résultats précis et sans erreur. Les seules limitations sont dictées par les spécifications matérielles et le temps QPU alloué, qui déterminent les volumes de circuits accessibles et les précisions de sortie. En revanche, de nombreuses solutions de réduction des erreurs sont spécifiques à une application ou font appel à des heuristiques non contrôlées, ce qui les rend inapplicables pour des circuits et des applications génériques.
-  **Jeu de portes étendu :** QESEM prend en charge les portes à angles fractionnaires et fournit des portes $Rzz(\theta)$ à angles fractionnaires optimisées par Qedma sur les dispositifs IBM Quantum Eagle. Ce jeu de portes étendu permet une compilation plus efficace et ouvre des volumes de circuits jusqu'à 2 fois plus grands par rapport à la compilation CX/CZ par défaut.
-	**Observables multi-bases :** QESEM prend en charge des observables d'entrée composées de nombreuses chaînes de Pauli non commutatives, comme les hamiltoniens génériques. Le choix des bases de mesure et l'optimisation de l'allocation des ressources QPU (shots et circuits) sont alors effectués automatiquement par QESEM pour minimiser le temps QPU requis pour la précision demandée. Cette optimisation, qui tient compte des fidélités matérielles et des taux d'exécution, te permet d'exécuter des circuits plus profonds et d'obtenir de meilleures précisions.
## Benchmarks
QESEM a été testé sur une grande variété de cas d'utilisation et d'applications. Les exemples suivants peuvent t'aider à évaluer quels types de charges de travail tu peux exécuter avec QESEM.

Un indicateur clé pour quantifier la difficulté tant de l'atténuation des erreurs que de la simulation classique pour un circuit et un observable donnés est le **volume actif** : le nombre de portes CNOT affectant l'observable dans le circuit. Le volume actif dépend de la profondeur et de la largeur du circuit, du poids de l'observable, et de la structure du circuit, qui détermine le cône de lumière de l'observable. Pour plus de détails, consulte l'exposé du [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM apporte une valeur particulièrement grande dans le régime de grand volume, en fournissant des résultats fiables pour des circuits et des observables génériques.

![Volume actif](../docs/images/guides/qedma-qesem/active_volume.svg)

| Application    | Nombre de qubits | Dispositif | Description du circuit | Précision | Temps total | Utilisation runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuit VQE  | 8              | Eagle (r3) | 21 couches au total, 9 bases de mesure, chaîne 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 couches uniques x 3 pas, topologie heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 couches uniques x 8 pas, topologie heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulation hamiltonienne de Trotter   | 40  | Eagle (r3)            | 2 couches uniques x 10 pas de Trotter, chaîne 1D                    | 97%      | 3 heures     | 25 min         |
| Simulation hamiltonienne de Trotter   | 119  | Eagle (r3)           | 3 couches uniques x 9 pas de Trotter, topologie heavy-hex 2D                    | 95%      | 6,5 heures     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 couches uniques x 15 pas, topologie heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

La précision est mesurée ici relativement à la valeur idéale de l'observable : $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, où « $\epsilon$ » est la précision absolue de l'atténuation (définie par l'entrée de l'utilisateur) et $\langle O \rangle_{ideal}$ est la valeur de l'observable pour le circuit sans bruit.
« Utilisation runtime » mesure l'utilisation du benchmark en mode batch (somme de l'utilisation des jobs individuels), tandis que « temps total » mesure l'utilisation en mode session (temps réel de l'expérience), ce qui inclut les temps de calcul classique et de communication supplémentaires. QESEM est disponible en exécution dans les deux modes, afin que tu puisses tirer le meilleur parti de tes ressources disponibles.

Les circuits Kicked Ising à 28 qubits simulent le quasi-cristal à temps discret étudié par Shinjo et al. (voir [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) et [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) sur trois boucles connectées d'ibm_kawasaki. Les paramètres de circuit utilisés ici sont $(\theta_x, \theta_z) = (0.9 \pi, 0)$, avec un état initial ferromagnétique $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. L'observable mesuré est la valeur absolue de la magnétisation $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. L'expérience Kicked Ising à l'échelle utilitaire a été exécutée sur les 136 meilleurs qubits d'ibm_fez ; ce benchmark particulier a été exécuté à l'angle de Clifford $(\theta_x, \theta_z) = (\pi, 0)$, auquel le volume actif croît lentement avec la profondeur du circuit — ce qui, combiné aux hautes fidélités du dispositif, permet d'atteindre une grande précision en peu de temps.

Les circuits de simulation hamiltonienne de Trotter correspondent à un modèle d'Ising en champ transverse à angles fractionnaires : $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ et $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ respectivement (voir [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Le circuit à l'échelle utilitaire a été exécuté sur les 119 meilleurs qubits d'ibm_brisbane, tandis que l'expérience à 40 qubits a été exécutée sur la meilleure chaîne disponible. La précision est rapportée pour la magnétisation ; des résultats de haute précision ont également été obtenus pour des observables de poids plus élevé.

Le circuit VQE a été développé en collaboration avec des chercheurs du Center for Quantum Technology and Applications du Deutsches Elektronen-Synchrotron (DESY). L'observable cible ici était un hamiltonien constitué d'un grand nombre de chaînes de Pauli non commutatives, ce qui met en valeur les performances optimisées de QESEM pour les observables multi-bases. L'atténuation a été appliquée à un ansatz optimisé classiquement ; bien que ces résultats ne soient pas encore publiés, des résultats de même qualité seront obtenus pour différents circuits ayant des propriétés structurelles similaires.
## Premiers pas
Authentifie-toi à l'aide de ta [clé API IBM Quantum Platform](http://quantum.cloud.ibm.com/), et sélectionne la Qiskit Function QESEM comme suit. (Cet extrait suppose que tu as déjà [sauvegardé ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Exemple
Pour commencer, essaie cet exemple de base qui estime le temps QPU nécessaire pour exécuter QESEM pour un `pub` donné :

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

L'exemple suivant exécute un job QESEM :

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Tu peux utiliser les API Qiskit Serverless habituelles pour vérifier le statut de ta charge de travail Qiskit Function ou récupérer les résultats :

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Paramètres de la fonction
| Nom |  Type | Description | Obligatoire | Défaut |  Exemple |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) | Il s'agit de l'entrée principale. Le `Pub` contient 2 à 4 éléments : un circuit, un ou plusieurs observables, 0 ou un seul ensemble de valeurs de paramètres, et une précision optionnelle. Si aucune précision n'est spécifiée, la valeur `default_precision` des `options` sera utilisée | Oui | N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string | Nom du backend à utiliser | Non | QESEM sélectionnera le dispositif le moins occupé signalé par IBM | `"ibm_fez"` |
| `instance` | string | Le nom de ressource cloud de l'instance à utiliser dans ce format | Non | N/A | `"CRN"` |
| `options` | dictionary | Options d'entrée. Voir la section **Options** pour plus de détails. | Non | Voir la section **Options** pour les détails. | `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}` |

### Options
| Option |  Choix | Description | Défaut |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"` / `"empirical"` / None  | Lorsque défini, le job QESEM ne calculera que l'estimation du temps QPU. Voir la description ci-dessous pour plus de détails. Lorsque défini à None, le circuit sera exécuté avec QESEM | None |
| `default_precision` | 0 < float | S'appliquera aux `pubs` qui n'ont pas de précision définie. La précision représente l'erreur acceptable sur les valeurs d'espérance des observables en valeur absolue. Autrement dit, le temps d'exécution QPU pour l'atténuation sera déterminé de façon à fournir des valeurs de sortie pour tous les observables d'intérêt qui se situent dans un intervalle de confiance `1σ` autour de la précision cible. Si plusieurs observables sont fournis, l'atténuation s'exécutera jusqu'à ce que la précision cible soit atteinte pour chacun des observables d'entrée. | 0.02 |
| `max_execution_time` | 0 < entier < 28 800 (8 heures) | Permet de limiter le temps QPU, exprimé en secondes, utilisé pour l'ensemble du processus QESEM. Voir les détails supplémentaires ci-dessous. | 3 600 (une heure) |
| `transpilation_level` | 0 / 1 | Voir la description ci-dessous | 1 |
| `execution_mode` | `"session"` / `"batch"` | Voir la description ci-dessous | "batch" |

> **Caution:** L'estimation du temps QPU varie d'un backend à l'autre. Par conséquent, lors de l'exécution de la fonction QESEM, assure-toi de l'exécuter sur le même backend que celui sélectionné lors de l'obtention de l'estimation du temps QPU.

> **Note:** QESEM terminera son exécution lorsqu'il atteindra la précision cible ou lorsqu'il atteindra `max_execution_time`, selon la première éventualité.

- `estimate_time_only` — Ce paramètre permet d'obtenir une estimation du temps QPU nécessaire pour exécuter le circuit avec QESEM.
    - S'il est défini à `"analytical"`, une borne supérieure du temps QPU est calculée sans consommer aucune ressource QPU. Cette estimation a une résolution de 30 minutes (par exemple, 30 minutes, 60 minutes, 90 minutes, etc.). Elle est généralement pessimiste, et ne peut être obtenue que pour des observables de Pauli uniques ou des sommes de Paulis sans supports qui se croisent (par exemple, Z0+Z1). Elle est principalement utile pour comparer les niveaux de complexité de différents paramètres fournis par l'utilisateur (circuit, précision, etc.).
    - Pour obtenir une estimation plus précise du temps QPU, définis ce paramètre à `"empirical"`. Bien que cette option nécessite l'exécution d'un petit nombre de circuits, elle fournit une estimation du temps QPU nettement plus précise. Cette estimation a une résolution de 5 minutes (par exemple, 20 minutes, 25 minutes, 30 minutes, etc.). Tu peux choisir d'exécuter l'estimation empirique du temps en mode batch ou en mode session. Pour plus de détails, consulte la description de `execution_mode`. Par exemple, en mode batch, l'estimation empirique du temps consommera moins de 10 minutes de temps QPU.

- `max_execution_time` : Permet de limiter le temps QPU, exprimé en secondes, utilisé pour l'ensemble du processus QESEM. Comme le temps QPU final nécessaire pour atteindre la précision cible est déterminé dynamiquement pendant le job QESEM, ce paramètre te permet de limiter le coût de l'expérience. Si le temps QPU déterminé dynamiquement est inférieur au temps alloué, ce paramètre n'affectera pas l'expérience. Le paramètre `max_execution_time` est particulièrement utile dans les cas où l'estimation analytique fournie par QESEM avant le démarrage du job est trop pessimiste et où tu souhaites quand même lancer un job d'atténuation. Une fois la limite de temps atteinte, QESEM cesse d'envoyer de nouveaux circuits. Les circuits déjà envoyés continuent de s'exécuter (le temps total peut donc dépasser la limite d'au plus 30 minutes), et tu reçois les résultats traités des circuits exécutés jusqu'à ce point. Si tu souhaites appliquer une limite de temps QPU inférieure à l'estimation analytique, consulte Qedma pour obtenir une estimation de la précision réalisable dans les limites de temps.

- `transpilation_level` : Après qu'un circuit est soumis à QESEM, il prépare automatiquement plusieurs transpilations alternatives du circuit et choisit celle qui minimise le temps QPU. Par exemple, les transpilations alternatives peuvent utiliser des portes RZZ fractionnaires optimisées par Qedma pour réduire la profondeur du circuit. Bien entendu, toutes les transpilations sont équivalentes au circuit d'entrée en termes de sortie idéale. Pour exercer un meilleur contrôle sur la transpilation du circuit, définis le niveau de transpilation dans les `options`. Alors que `"transpilation_level": 1` correspond au comportement par défaut décrit ci-dessus, `"transpilation_level": 0` n'inclut que les modifications minimales requises par rapport au circuit original ; par exemple, la « layerification » — l'organisation des opérations du circuit en « couches » de portes à deux qubits simultanées. Note que le mapping automatique sur les qubits de haute fidélité du matériel est appliqué dans tous les cas.

| transpilation_level | Description |
|:-:|:--|
| `1` | Transpilation QESEM par défaut. Prépare plusieurs transpilations alternatives et choisit celle qui minimise le temps QPU. Les barrières peuvent être modifiées lors de l'étape de layerification. |
| `0` | Transpilation minimale : le circuit atténué ressemblera étroitement au circuit d'entrée en termes de structure. Les circuits fournis au niveau 0 doivent correspondre à la connectivité du dispositif et être spécifiés en termes des portes suivantes : CX, Rzz(α), et les portes standard à un qubit (U, x, sx, rz, etc.). Les barrières seront respectées lors de l'étape de layerification. |

- `execution_mode` — Tu peux choisir d'exécuter le job QESEM dans une [session IBM](/guides/execution-modes#session-mode) dédiée ou sur plusieurs [batches IBM](/guides/execution-modes#batch-mode) :
    -   **Mode Session** : Les sessions sont plus coûteuses mais permettent un temps de résultat plus court. Une fois la session démarrée, le QPU est réservé exclusivement au job QESEM. Le calcul d'utilisation inclut à la fois le temps passé en exécution QPU et les calculs classiques associés (effectués par QESEM et IBM). La Qiskit Function QESEM prend en charge la création et la fermeture de la session automatiquement. Pour les utilisateurs disposant d'un accès illimité aux QPU (par exemple, les configurations sur site), il est recommandé d'utiliser le mode session pour une exécution QESEM plus rapide.
    -   **Mode Batch** : En mode batch, le QPU est libéré pendant les calculs classiques, ce qui conduit à une utilisation QPU moindre. Comme les jobs batch couvrent généralement une période plus longue, le risque de dérives matérielles est plus élevé ; QESEM intègre des mécanismes pour détecter et compenser ces dérives, maintenant ainsi la fiabilité sur des exécutions prolongées.

> **Note:** Les opérations de barrière sont généralement utilisées pour spécifier les couches de portes à deux qubits dans les circuits quantiques. Au niveau 0, QESEM préserve les couches spécifiées par les barrières. Au niveau 1, les couches spécifiées par les barrières sont considérées comme une alternative de transpilation lors de la minimisation du temps QPU.
### Sorties
La sortie d'une Circuit function est un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), qui contient deux champs :

- Un objet [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Il peut être indexé directement depuis le `PrimitiveResult`.

- Des métadonnées au niveau du job.

Chaque `PubResult` contient un champ `data` et un champ `metadata`.

- Le champ `data` contient au moins un tableau de valeurs d'espérance (`PubResult.data.evs`) et un tableau d'erreurs standard (`PubResult.data.stds`). Il peut également contenir davantage de données selon les options utilisées.

- Le champ `metadata` contient des métadonnées au niveau PUB (`PubResult.metadata`).

L'extrait de code suivant décrit comment récupérer l'estimation du temps QPU (lorsque `estimate_time_only` est défini) :

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

L'extrait de code suivant montre comment récupérer les résultats d'atténuation (lorsque `estimate_time_only` n'est pas défini) et les métriques d'exécution. Ces données contiennent des informations essentielles permettant de mieux comprendre comment différents paramètres influencent l'exécution de QESEM. Elles peuvent également être utiles lors de la rédaction d'un article basé sur tes recherches.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Récupérer les messages d'erreur
Si le statut de ta charge de travail est ERROR, utilise job.result() pour récupérer le message d'erreur comme suit :

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>